# Rice Yield Forecasting System — Data Analysis

This notebook explores the rice yield dataset, checks data quality, summarizes trends, and creates visualizations.

**Expected dataset:** `data/rice_dataset.csv`

> If the dataset is synthetic/simulated, report findings as demonstration results, not official agricultural statistics.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "rice_dataset.csv").exists() and (PROJECT_ROOT.parent / "data" / "rice_dataset.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "rice_dataset.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Place rice_dataset.csv in the project's data/ folder.")

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())


## 1. Dataset overview
Review the column names, data types, and summary statistics.

In [ ]:
print("Columns:", df.columns.tolist())
df.info()
display(df.describe(include="all").T)


## 2. Data quality checks
Check missing values, duplicate records, and infinite numeric values.

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)
display(quality)
print(f"Duplicate rows: {df.duplicated().sum():,}")

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)
    display(pd.Series(np.isinf(df[numeric_cols]).sum(), name="infinite_count"))


## 3. Target variable: rice yield
The target column is `yield_kg_per_ha` (kilograms per hectare).

In [ ]:
TARGET = "yield_kg_per_ha"
if TARGET not in df.columns:
    raise KeyError(f"Required target column {TARGET!r} is missing.")

display(df[TARGET].describe().to_frame("yield_kg_per_ha"))
plt.figure(figsize=(9, 5))
plt.hist(df[TARGET].dropna(), bins=30, edgecolor="black")
plt.title("Distribution of Rice Yield")
plt.xlabel("Yield (kg/ha)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


## 4. Yield trends over time
Plot average yield by year if a `year` column is available.

In [ ]:
if "year" in df.columns:
    yearly_yield = df.groupby("year", as_index=False)[TARGET].mean().sort_values("year")
    display(yearly_yield)
    plt.figure(figsize=(10, 5))
    plt.plot(yearly_yield["year"], yearly_yield[TARGET], marker="o")
    plt.title("Average Rice Yield by Year")
    plt.xlabel("Year")
    plt.ylabel("Average yield (kg/ha)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No year column found; skipping yearly trend chart.")


## 5. Compare yield by state and season
These are descriptive comparisons; group sizes can affect the results.

In [ ]:
if "state" in df.columns:
    state_yield = df.groupby("state", as_index=False)[TARGET].mean().sort_values(TARGET, ascending=False)
    display(state_yield.head(15))
    plot_data = state_yield.head(15).sort_values(TARGET)
    plt.figure(figsize=(11, 6))
    plt.barh(plot_data["state"].astype(str), plot_data[TARGET])
    plt.title("Top 15 States by Average Rice Yield")
    plt.xlabel("Average yield (kg/ha)")
    plt.tight_layout()
    plt.show()

if "season" in df.columns:
    season_yield = df.groupby("season", as_index=False)[TARGET].mean().sort_values(TARGET, ascending=False)
    display(season_yield)
    plt.figure(figsize=(9, 5))
    plt.bar(season_yield["season"].astype(str), season_yield[TARGET])
    plt.title("Average Rice Yield by Season")
    plt.xlabel("Season")
    plt.ylabel("Average yield (kg/ha)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 6. Correlation analysis
Correlation indicates linear association, not causation.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    display(corr.round(2))
    fig, ax = plt.subplots(figsize=(11, 8))
    image = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title("Correlation Matrix of Numeric Features")
    fig.colorbar(image, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for correlation analysis.")


## 7. Feature leakage checks
Review columns that directly reveal the target or may not be available at forecast time.

In [ ]:
excluded_candidates = [c for c in ["yield_kg_per_ha", "production_tonnes", "data_type"] if c in df.columns]
print("Columns to review/exclude from model inputs:", excluded_candidates)
if "production_tonnes" in df.columns and "area_hectares" in df.columns:
    print("Warning: production_tonnes and area_hectares may mathematically determine yield.")
    print("Avoid production_tonnes as an input when forecasting before harvest.")
if "previous_year_yield_kg_per_ha" in df.columns:
    print("Previous-year yield may be useful if it is available at prediction time.")


## 8. Export analysis summaries
Save reusable summary CSV files into the project's `reports/` folder.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
quality.to_csv(REPORTS_DIR / "data_quality_report.csv")

if "year" in df.columns:
    df.groupby("year", as_index=False)[TARGET].mean().sort_values("year").to_csv(
        REPORTS_DIR / "yearly_yield_summary.csv", index=False
    )
if "state" in df.columns:
    df.groupby("state", as_index=False)[TARGET].mean().sort_values(TARGET, ascending=False).to_csv(
        REPORTS_DIR / "state_yield_summary.csv", index=False
    )
print(f"Analysis summaries saved to: {REPORTS_DIR}")


## Conclusion template

After running the notebook, summarize the yield distribution, year-to-year trend, state/season differences, missing values, and strongest correlations. Avoid causal claims from correlations. If the dataset is synthetic, clearly label findings as simulated rather than official agricultural estimates.
